# Chess Rating Systems: Building and Comparing Elo & Glicko-2

**Author:** Atilla Ahmed.
**Data Sources:** Lichess Game Database (Kaggle) · FIDE Official Rating Lists

## Table of Content

1. Introduction
2. Data Loading & Initial Inspection
3. Data Cleaning & Preprocessing
4. Exploratory Data Analysis
5. Mathematical Framework
6. Implementation
7. Hypothesis Testing
8. Conclusion
9. Self-Assessment

---

## 1. Introduction

### 1.1 Motivation

I play chess online and I've always been curious about what the rating number actually means mathematically. The **Elo system**, introduced in the 1960s by Arpad Elo and adopted by FIDE in 1970, uses a logistic function to estimate the probability that one player beats another based on their rating difference. It's simple and widely used, but it has a weakness it doesn't track how *certain* the system is about a player's rating.

**Glicko-2** (Glickman, 2001) solves this by adding a rating deviation (uncertainty) and a volatility parameter per player. Lichess uses Glicko-2; FIDE still uses Elo. In this project I derive both systems from scratch, implement them in Python, and test them on real data.

### 1.2 Problem Statement & Objectives

> **How well do Elo and Glicko-2 predict chess outcomes, and what do online (Lichess) and over-the-board (FIDE) data reveal about how ratings behave in practice?**

Specific questions I want to answer:

1. How well-calibrated is the Elo prediction? (If the model says 70% win chance, does white win ~70% of the time?)
2. Is white's first-move advantage statistically significant, and does it vary by rating level?
3. Do certain openings produce significantly different win rates?
4. How do Lichess and FIDE rating distributions compare structurally?
5. Is there a gender gap in FIDE ratings, and can participation rates explain it?
6. Can a logistic regression model that includes color, opening, and time control outperform pure Elo?

Each of these is formalized as a hypothesis test or model comparison in Sections 6–7.

The project has three goals: derive the math behind Elo and Glicko-2 with full formulas, implement both from scratch (no pre-built rating libraries), and run statistical analysis on real data from two independent sources.

### 1.3 Data Sources

**Lichess (Kaggle)** - 20,000+ online games in CSV format. Each row is a game with both players' ratings, the result, opening name/ECO code, number of moves, and time control. Licensed CC0.

**FIDE (ratings.fide.com)** - official rating list in fixed-width TXT. Each row is a player with Standard/Rapid/Blitz ratings, title, federation, gender, and birth year.

The two datasets can't be merged at the player level (no shared ID), but I compare them at the aggregate level, distribution shapes, population parameters, and how the Elo formula performs in each context.

### 1.4 Prior Work

Elo described his system in *The Rating of Chessplayers, Past and Present* (1978). Glickman published Glicko-2 in 2001 — his paper *"Example of the Glicko-2 System"* is the main reference I follow for the implementation. There's also ML-based approaches (logistic regression, gradient boosting) that can outperform Elo in raw accuracy, but I'm more interested in understanding the mathematical structure than maximizing a metric.

## 2. Data Loading and Initial Inspection
This section Loads both datasets, check their structure, and flags anything that need attention before the real analysis. The Lichess data comes as a standard CSV with one row per game. The FIDE data is a fixed-width text file a snapshot of every registered player with their current ratings.

### 2.0 Setup and Imports

In [1]:
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=RuntimeWarning)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
from scipy import stats
from scipy.stats import skew, kurtosis, shapiro, binomtest

### 2.1 Loading the Lichess Dataset
The Lichess data is a CSV from Kaggle containing ~20,000 games played.


In [2]:
lichess = pd.read_csv('games.csv')
# Let's look the shape
print(f'Shape: {lichess.shape[0]:,} rows x {lichess.shape[1]:,} columns ')
print()

# Columns types and non-null counts
lichess.info()

Shape: 20,058 rows x 16 columns 

<class 'pandas.DataFrame'>
RangeIndex: 20058 entries, 0 to 20057
Data columns (total 16 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   id              20058 non-null  str    
 1   rated           20058 non-null  bool   
 2   created_at      20058 non-null  float64
 3   last_move_at    20058 non-null  float64
 4   turns           20058 non-null  int64  
 5   victory_status  20058 non-null  str    
 6   winner          20058 non-null  str    
 7   increment_code  20058 non-null  str    
 8   white_id        20058 non-null  str    
 9   white_rating    20058 non-null  int64  
 10  black_id        20058 non-null  str    
 11  black_rating    20058 non-null  int64  
 12  moves           20058 non-null  str    
 13  opening_eco     20058 non-null  str    
 14  opening_name    20058 non-null  str    
 15  opening_ply     20058 non-null  int64  
dtypes: bool(1), float64(2), int64(4), str(9)
memory usage

In [3]:
# Preview first 5 rows
lichess.head()

,id,rated,created_at,last_move_at,turns,victory_status,winner,increment_code,white_id,white_rating,black_id,black_rating,moves,opening_eco,opening_name,opening_ply
0,TZJHLljE,False,1.504210e+12,1.504210e+12,13,outoftime,white,15+2,bourgris,1500,a-00,1191,d4 d5 c4 c6 cxd5 e6 dxe6 fxe6 Nf3 Bb4+ Nc3 Ba5...,D10,Slav Defense: Exchange Variation,5
1,l1NXvwaE,True,1.504130e+12,1.504130e+12,16,resign,black,5+10,a-00,1322,skinnerua,1261,d4 Nc6 e4 e5 f4 f6 dxe5 fxe5 fxe5 Nxe5 Qd4 Nc6...,B00,Nimzowitsch Defense: Kennedy Variation,4
2,mIICvQHh,True,1.504130e+12,1.504130e+12,61,mate,white,5+10,ischia,1496,a-00,1500,e4 e5 d3 d6 Be3 c6 Be2 b5 Nd2 a5 a4 c5 axb5 Nc...,C20,King's Pawn Game: Leonardis Variation,3
3,kWKvrqYL,True,1.504110e+12,1.504110e+12,61,mate,white,20+0,daniamurashov,1439,adivanov2009,1454,d4 d5 Nf3 Bf5 Nc3 Nf6 Bf4 Ng4 e3 Nc6 Be2 Qd7 O...,D02,Queen's Pawn Game: Zukertort Variation,3
4,9tXo1AUZ,True,1.504030e+12,1.504030e+12,95,mate,white,30+3,nik221107,1523,adivanov2009,1469,e4 e5 Nf3 d6 d4 Nc6 d5 Nb4 a3 Na6 Nc3 Be7 b4 N...,C41,Philidor Defense,5
